# Market Basket Analysis Notebook

This notebook mirrors the production pipeline: preprocessing, EDA, Apriori, FP-Growth, rule generation, and recommendation checks.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from scripts.run_analysis import MIN_ITEM_FREQUENCY, pairwise_statistics
from src.models import compare_algorithms, mine_rules, prune_redundant_rules, serialize_itemsets
from src.preprocessing import (
    clean_groceries_data,
    co_occurrence_matrix,
    create_basket_matrix,
    item_frequency,
    monthly_transaction_counts,
)
from src.recommend import recommend_items

In [ ]:
raw_df = pd.read_csv(PROJECT_ROOT / "data" / "groceries.csv")
cleaned_df = clean_groceries_data(raw_df)
basket_df = create_basket_matrix(cleaned_df, min_item_frequency=MIN_ITEM_FREQUENCY)

summary = pd.DataFrame(
    {
        "metric": ["rows", "transactions", "products", "average_basket_size"],
        "value": [
            len(cleaned_df),
            len(basket_df),
            basket_df.shape[1],
            round(basket_df.sum(axis=1).mean(), 2),
        ],
    }
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
item_frequency(basket_df).head(15).sort_values().plot(kind="barh", ax=axes[0], title="Top Items")
basket_df.sum(axis=1).plot(kind="hist", bins=20, ax=axes[1], title="Transaction Size Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(co_occurrence_matrix(basket_df, top_n=20), cmap="YlGnBu")
plt.title("Top Item Co-occurrence Heatmap")
plt.show()

monthly_transaction_counts(cleaned_df).plot(figsize=(10, 4), marker="o", title="Monthly Transaction Trend")
plt.ylabel("Transactions")
plt.show()

In [ ]:
pair_stats = pairwise_statistics(basket_df)
pair_stats.head(10)

In [ ]:
comparison = compare_algorithms(basket_df, support_values=(0.01, 0.02, 0.05), min_confidence=0.5, min_lift=1.2)
comparison

In [ ]:
apriori_itemsets, apriori_rules = mine_rules(
    basket_df,
    algorithm="apriori",
    min_support=0.001,
    min_confidence=0.05,
    min_lift=1.2,
)
fpgrowth_itemsets, fpgrowth_rules = mine_rules(
    basket_df,
    algorithm="fpgrowth",
    min_support=0.001,
    min_confidence=0.05,
    min_lift=1.2,
)

rules = serialize_itemsets(prune_redundant_rules(fpgrowth_rules))
rules[["antecedents", "consequents", "support", "confidence", "lift", "leverage", "conviction"]].head(10)

In [ ]:
_, strict_rules = mine_rules(
    basket_df,
    algorithm="fpgrowth",
    min_support=0.001,
    min_confidence=0.5,
    min_lift=1.2,
)
print(f"Strict actionable rules: {len(strict_rules)}")

In [ ]:
recommend_items(["whole milk", "yogurt"], rules, top_n=5, min_confidence=0.05, min_lift=1.2)